In [2]:
import pandas as pd
from pathlib import Path

DATA = Path.home() / "datasets" / "apache-jira" / "issues.csv"
print("File exists:", DATA.exists())
print("Size (GB):", round(DATA.stat().st_size / 1e9, 2))



File exists: True
Size (GB): 1.91


In [3]:
cols = [
    "key", "created", "resolutiondate",
    "priority.name", "status.name", "statusCategory.name",
    "resolution.name", "issuetype.name",
    "project.key", "project.name",
]

df = pd.read_csv(DATA, usecols=cols)
print(df.shape)
df.head()

(1149323, 10)


,key,resolution.name,priority.name,status.name,statusCategory.name,issuetype.name,project.key,project.name,resolutiondate,created
0,WW-712,Fixed,Minor,Closed,Done,Improvement,WW,Struts 2,2005-01-01 07:50:46,2005-01-01 07:47:50
1,XALANC-446,Fixed,Blocker,Resolved,Done,Bug,XALANC,XalanC,2004-12-30 05:30:36,2004-12-25 22:50:30
2,ROL-587,Cannot Reproduce,Critical,Closed,Done,Bug,ROL,Apache Roller,2005-01-02 15:21:00,2005-01-01 13:52:46
3,AXIS-1741,NaN,Major,Open,To Do,Bug,AXIS,Axis,NaN,2005-01-02 19:13:37
4,AXIS-1745,NaN,Major,Open,To Do,Bug,AXIS,Axis,NaN,2005-01-03 03:34:52


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1149323 entries, 0 to 1149322
Data columns (total 10 columns):
 #   Column               Non-Null Count    Dtype
---  ------               --------------    -----
 0   key                  1149323 non-null  str  
 1   resolution.name      946655 non-null   str  
 2   priority.name        1117464 non-null  str  
 3   status.name          1131224 non-null  str  
 4   statusCategory.name  1131224 non-null  str  
 5   issuetype.name       1131224 non-null  str  
 6   project.key          1131224 non-null  str  
 7   project.name         1131224 non-null  str  
 8   resolutiondate       946715 non-null   str  
 9   created              1131224 non-null  str  
dtypes: str(10)
memory usage: 87.7 MB


In [5]:
df["created"] = pd.to_datetime(df["created"], errors="coerce")
df["resolutiondate"] = pd.to_datetime(df["resolutiondate"], errors="coerce")

df[["created", "resolutiondate"]].info()

<class 'pandas.DataFrame'>
RangeIndex: 1149323 entries, 0 to 1149322
Data columns (total 2 columns):
 #   Column          Non-Null Count    Dtype         
---  ------          --------------    -----         
 0   created         1130935 non-null  datetime64[us]
 1   resolutiondate  946715 non-null   datetime64[us]
dtypes: datetime64[us](2)
memory usage: 17.5 MB


In [6]:
df["resolution_days"] = (df["resolutiondate"] - df["created"]).dt.total_seconds() / 86400

df["resolution_days"].describe()

count    946426.000000
mean        203.954379
std         520.375742
min          -0.372951
25%           1.049771
50%          11.915307
75%         112.017925
max        8001.517257
Name: resolution_days, dtype: float64

In [7]:
(df["resolution_days"] < 0).sum()

np.int64(18)

In [8]:
resolved = df[df["resolution_days"] >= 0].copy()
print("Resolved issues for analysis:", len(resolved))
print("Dropped (negative):", (df["resolution_days"] < 0).sum())

Resolved issues for analysis: 946408
Dropped (negative): 18


In [9]:
resolved.groupby("priority.name")["resolution_days"].median().sort_values()

priority.name
Trivial             4.576389
P0                  4.814670
Urgent              6.900596
Blocker             7.770035
Low                 9.627245
Major              10.938970
P4                 11.450741
Critical           11.963102
High               12.830822
Normal             13.292616
Minor              14.471557
P1                 28.692049
P2                 28.863605
P3                 98.932072
Not a Priority    746.897992
Name: resolution_days, dtype: float64

In [11]:
resolved["project.name"].value_counts()

project.name
Spark                                   44867
Apache Flex                             31565
Flink                                   30386
HBase                                   26611
Ambari                                  23649
                                        ...  
eSCIMo                                      1
Axis2 Transports                            1
ActiveMQ Website                            1
COTTON                                      1
Maven JLink Plugin (Moved to GitHub)        1
Name: count, Length: 631, dtype: int64

In [12]:
resolved["priority.name"].value_counts()

priority.name
Major             620202
Minor             179388
Critical           44155
Blocker            34793
Trivial            27202
Normal             11227
P2                  6390
Low                 6164
P3                  2629
P1                   721
Urgent               642
Not a Priority       354
P0                   274
P4                   237
High                 156
Name: count, dtype: int64

In [15]:
# For each project, get the set of distinct priorities it uses
priority_menus = resolved.groupby("project.name")["priority.name"].unique()

# Look at a few big projects' menus
for proj in ["Spark", "Apache Flex", "Flink", "HBase", "Ambari"]:
    print(proj, "->", sorted(priority_menus[proj]))

Spark -> ['Blocker', 'Critical', 'Major', 'Minor', 'Trivial']
Apache Flex -> ['Blocker', 'Critical', 'Major', 'Minor', 'Trivial']
Flink -> ['Blocker', 'Critical', 'Major', 'Minor', 'Not a Priority']
HBase -> ['Blocker', 'Critical', 'Major', 'Minor', 'Trivial']
Ambari -> ['Blocker', 'Critical', 'Major', 'Minor', 'Trivial']


In [16]:
# How many projects include each of the four core priorities?
core = ["Blocker", "Critical", "Major", "Minor"]
for p in core:
    n = resolved[resolved["priority.name"] == p]["project.name"].nunique()
    print(f"{p}: used by {n} projects")

Blocker: used by 484 projects
Critical: used by 501 projects
Major: used by 614 projects
Minor: used by 567 projects
